## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 
>
> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.

Centola’s study relies on custom made data collected through a controlled online experiment. A key advantage is strong causal identification. The researcher designs the network structure, controls exposure, and isolates reinforcement effects, which makes it possible to directly test complex contagion. This fits the lecture’s point that experiments allow clear cause and effect. The drawbacks are high cost, smaller samples, and artificial settings, which limit external validity and realism.

Nicolaides’s study uses ready made observational data from fitness tracking apps. Its strengths are large scale, long time coverage, and real world behavior, which increase external validity and allow the study of behavior over time. This matches the lecture’s discussion of big data as always on and nonreactive. The disadvantages are limited control over data generation and risks of confounding and homophily. Causal claims require strong assumptions and advanced methods, such as instrumental variables.

> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

The differences in data sources strongly affect how the results should be interpreted. In Centola’s study, the use of custom made experimental data allows a clear causal interpretation. Differences in behavior can be directly attributed to network structure and social reinforcement, because the environment is controlled. However, the artificial setting means the results may not fully generalize to real world social networks.

In Nicolaides’s study, the use of ready made observational data captures real world behavior at large scale, which improves external validity. The results are more representative of how people actually behave in daily life. However, causal interpretation is weaker. Even with advanced methods like instrumental variables, the findings depend on modeling assumptions and may still be affected by unobserved confounding. As a result, the results should be interpreted as evidence of influence under specific assumptions rather than definitive proof of causality.

## Part 2: Find Researchers using the OpenAlex API

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
> **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__: 
> 
> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     

Dataset D2 contains 15,125 unique works. These were retrieved from IC2S2 authors after applying filters on author productivity, citation count, team size, and disciplinary relevance. To estimate the number of unique researchers, we aggregated all author_ids across works and removed duplicates. This resulted in a substantially larger set than the original IC2S2 seed authors, reflecting the collaborative nature of the field. The dataset captures a broad but filtered view of Computational Social Science publications and their co-authorship structure. The equal number of rows in D2 and D3 confirms structural consistency between metadata and abstracts.

> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?    

We implemented bulk queries by batching up to 25 authors per request and applied filters directly in the API call. We increased per-page to 200 and used cursor-based pagination to retrieve complete result sets. Deduplication was handled with a set of seen work IDs. Checkpoint saving reduced risk of data loss. These strategies reduced the number of API calls and avoided unnecessary post-processing. Efficient code is crucial when scaling to large datasets. Execution time was significantly lower compared to naive per-author querying.

> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?    

The thresholds improve data quality and focus. Limiting authors to 5–5000 works removes marginal or anomalous profiles. Requiring more than 10 citations filters out low-impact publications. Restricting authorships to fewer than 10 reduces inclusion of large consortium papers. Filtering by level-0 Concepts ensures alignment with Computational Social Science while intersecting with quantitative disciplines. These choices increase thematic coherence and manage dataset size. However, they may underrepresent early-career scholars, interdisciplinary work outside predefined concepts, or large collaborative research common in physics or computational biology. The dataset therefore emphasizes established, quantitatively oriented CSS research.

In [11]:
import requests, csv, re, unicodedata
from io import StringIO

CSV_URL = "https://ic2s2-2025.org/files/ic2s2_2025_schedule_v5.csv"

def clean(s):
    return re.sub(r"\s+", " ", (s or "").replace("\u00a0", " ").strip())

def split_names(s):
    s = clean(s)
    if not s:
        return []
    s = s.replace("\n", ",")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return [p for p in parts if len(p.split()) >= 2]

def norm_key(name):
    name = clean(name).lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-z\s\-']", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

r = requests.get(CSV_URL, timeout=60)
r.raise_for_status()
rows = list(csv.DictReader(StringIO(r.text)))

all_people = []
for row in rows:
    all_people += split_names(row.get("author", ""))
    all_people += split_names(row.get("chair", ""))

unique = {}
for person in all_people:
    k = norm_key(person)
    if k and k not in unique:
        unique[k] = person

final_names = sorted(unique.values(), key=lambda x: norm_key(x))

out_path = "ic2s2_2025_participants.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name"])
    for n in final_names:
        w.writerow([n])

bad = [n for n in final_names if "," in n]
print("Unikke personer:", len(final_names))
print("Rækker der stadig indeholder komma:", len(bad))
print("Gemte fil:", out_path)

Unikke personer: 1559
Rækker der stadig indeholder komma: 0
Gemte fil: ic2s2_2025_participants.csv


In [6]:
import pandas as pd
import requests
import time
import random

NAMES_CSV = "ic2s2_2025_participants.csv"
OUT_CSV = "ic2s2_2025_openalex_authors.csv"

MAILTO = "asghar.menashi@gmail.com"  # put your real email

names = pd.read_csv(NAMES_CSV)["name"].dropna().tolist()

BASE_URL = "https://api.openalex.org/authors"

def fetch_author(name, max_retries=5):
    for attempt in range(max_retries):
        r = requests.get(
            BASE_URL,
            params={"search": name, "per-page": 1, "mailto": MAILTO},
            timeout=30
        )

        # rate limit or temporary error
        if r.status_code in (429, 500, 502, 503, 504):
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
            continue

        if r.status_code != 200:
            return {"status": r.status_code, "openalex_id": None, "display_name": None,
                    "works_api_url": None, "h_index": None, "works_count": None, "country_code": None}

        data = r.json()

        if "results" not in data or not data["results"]:
            return {"status": 200, "openalex_id": None, "display_name": None,
                    "works_api_url": None, "h_index": None, "works_count": None, "country_code": None}

        a = data["results"][0]
        return {
            "status": 200,
            "openalex_id": a.get("id"),
            "display_name": a.get("display_name"),
            "works_api_url": a.get("works_api_url"),
            "h_index": a.get("summary_stats", {}).get("h_index"),
            "works_count": a.get("works_count"),
            "country_code": a.get("last_known_institution", {}).get("country_code"),
        }

    # gave up after retries
    return {"status": 429, "openalex_id": None, "display_name": None,
            "works_api_url": None, "h_index": None, "works_count": None, "country_code": None}

rows = []
for i, name in enumerate(names, start=1):
    info = fetch_author(name)
    rows.append({"searched_name": name, **info})

    if i % 25 == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
        print(f"Saved {i}/{len(names)}")

    time.sleep(0.25)  # stable pace

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print("Done.")
print("Total:", len(df))
print("Matched:", df["openalex_id"].notna().sum())
print("Unmatched:", df["openalex_id"].isna().sum())

Saved 25/1559
Saved 50/1559
Saved 75/1559
Saved 100/1559
Saved 125/1559
Saved 150/1559
Saved 175/1559
Saved 200/1559
Saved 225/1559
Saved 250/1559
Saved 275/1559
Saved 300/1559
Saved 325/1559
Saved 350/1559
Saved 375/1559
Saved 400/1559
Saved 425/1559
Saved 450/1559
Saved 475/1559
Saved 500/1559
Saved 525/1559
Saved 550/1559
Saved 575/1559
Saved 600/1559
Saved 625/1559
Saved 650/1559
Saved 675/1559
Saved 700/1559
Saved 725/1559
Saved 750/1559
Saved 775/1559
Saved 800/1559
Saved 825/1559
Saved 850/1559
Saved 875/1559
Saved 900/1559
Saved 925/1559
Saved 950/1559
Saved 975/1559
Saved 1000/1559


KeyboardInterrupt: 

In [ ]:
# It was late and got tired so i paused it and went to sleep. Then i resumed it the next day.

In [1]:
import pandas as pd
import requests
import time
import random
import os

NAMES_CSV = "ic2s2_2025_participants.csv"
OUT_CSV = "ic2s2_2025_openalex_authors.csv"
MAILTO = "asghar.menashi@gmail.com"

names = pd.read_csv(NAMES_CSV)["name"].dropna().tolist()

# Load existing progress if file exists
if os.path.exists(OUT_CSV):
    existing_df = pd.read_csv(OUT_CSV)
    rows = existing_df.to_dict("records")
    start_index = len(existing_df)
    print(f"Resuming from {start_index}/{len(names)}")
else:
    rows = []
    start_index = 0
    print("Starting from scratch")

BASE_URL = "https://api.openalex.org/authors"

def fetch_author(name, max_retries=6):
    for attempt in range(max_retries):
        try:
            r = requests.get(
                BASE_URL,
                params={"search": name, "per-page": 1, "mailto": MAILTO},
                timeout=20
            )
        except requests.exceptions.RequestException:
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
            continue

        if r.status_code in (429, 500, 502, 503, 504):
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
            continue

        if r.status_code != 200:
            return {"status": r.status_code, "openalex_id": None, "display_name": None,
                    "works_api_url": None, "h_index": None, "works_count": None, "country_code": None}

        data = r.json()
        if "results" not in data or not data["results"]:
            return {"status": 200, "openalex_id": None, "display_name": None,
                    "works_api_url": None, "h_index": None, "works_count": None, "country_code": None}

        a = data["results"][0]
        return {
            "status": 200,
            "openalex_id": a.get("id"),
            "display_name": a.get("display_name"),
            "works_api_url": a.get("works_api_url"),
            "h_index": a.get("summary_stats", {}).get("h_index"),
            "works_count": a.get("works_count"),
            "country_code": a.get("last_known_institution", {}).get("country_code"),
        }

    return {"status": 429, "openalex_id": None, "display_name": None,
            "works_api_url": None, "h_index": None, "works_count": None, "country_code": None}

# Resume loop
for i in range(start_index, len(names)):
    name = names[i]
    print(f"Working on {i+1}/{len(names)}: {name}")

    info = fetch_author(name)
    rows.append({"searched_name": name, **info})

    if (i + 1) % 25 == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
        print(f"Saved {i+1}/{len(names)}")

    time.sleep(0.25)

pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

print("Finished.")
print("Total rows:", len(rows))

Resuming from 1000/1559
Working on 1001/1559: Michael Quayle
Working on 1002/1559: Michael S. Bernstein
Working on 1003/1559: Michael Szell
Working on 1004/1559: Michael Tiemann
Working on 1005/1559: Michael Yeomans
Working on 1006/1559: Michail Chatzianastasis
Working on 1007/1559: Michalina Janik
Working on 1008/1559: Michalis Vazirgiannis
Working on 1009/1559: Michele d'Errico
Working on 1010/1559: Michele Starnini
Working on 1011/1559: MICHELE TIZZANI
Working on 1012/1559: Michele Tizzoni
Working on 1013/1559: Michelle Horváth
Working on 1014/1559: Miguel Escobar Varela
Working on 1015/1559: Mihály Fazekas
Working on 1016/1559: Miia Bask
Working on 1017/1559: Mikael Bask
Working on 1018/1559: Mike Conway
Working on 1019/1559: Mikhail Tamm
Working on 1020/1559: Mikko Kivelä
Working on 1021/1559: Mila Oiva
Working on 1022/1559: Milena Tsvetkova
Working on 1023/1559: Miner Ye
Working on 1024/1559: Mingmeng Geng
Working on 1025/1559: Mingmin Feng
Saved 1025/1559
Working on 1026/1559: M

## Part 3: Collect Research Articles

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
> **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__: 
> 
> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     

Dataset D2 contains 15,125 unique works. These were retrieved from IC2S2 authors after applying filters on author productivity, citation count, team size, and disciplinary relevance. To estimate the number of unique researchers, we aggregated all author_ids across works and removed duplicates. This resulted in a substantially larger set than the original IC2S2 seed authors, reflecting the collaborative nature of the field. The dataset captures a broad but filtered view of Computational Social Science publications and their co-authorship structure. The equal number of rows in D2 and D3 confirms structural consistency between metadata and abstracts.

> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?    

We implemented bulk queries by batching up to 25 authors per request and applied filters directly in the API call. We increased per-page to 200 and used cursor-based pagination to retrieve complete result sets. Deduplication was handled with a set of seen work IDs. Checkpoint saving reduced risk of data loss. These strategies reduced the number of API calls and avoided unnecessary post-processing. Efficient code is crucial when scaling to large datasets. Execution time was significantly lower compared to naive per-author querying.

> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?    

The thresholds improve data quality and focus. Limiting authors to 5–5000 works removes marginal or anomalous profiles. Requiring more than 10 citations filters out low-impact publications. Restricting authorships to fewer than 10 reduces inclusion of large consortium papers. Filtering by level-0 Concepts ensures alignment with Computational Social Science while intersecting with quantitative disciplines. These choices increase thematic coherence and manage dataset size. However, they may underrepresent early-career scholars, interdisciplinary work outside predefined concepts, or large collaborative research common in physics or computational biology. The dataset therefore emphasizes established, quantitatively oriented CSS research.

In [9]:
import pandas as pd
import requests
import time
import random
import json
import math

MAILTO = "asghar.menashi@gmail.com"

D1_AUTHORS_CSV = "ic2s2_2025_openalex_authors.csv"
OUT_D2_PAPERS_CSV = "ic2s2_D2_papers.csv"
OUT_D3_ABSTRACTS_CSV = "ic2s2_D3_abstracts.csv"

WORKS_ENDPOINT = "https://api.openalex.org/works"
CONCEPTS_ENDPOINT = "https://api.openalex.org/concepts"

def request_json(url, params, max_retries=7, timeout=30):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
        except requests.exceptions.RequestException:
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
            continue
        if r.status_code in (429, 500, 502, 503, 504):
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
            continue
        if r.status_code != 200:
            return None, r.status_code
        try:
            return r.json(), 200
        except Exception:
            return None, 200
    return None, 429

def get_concept_id(display_name):
    params = {"search": display_name, "per-page": 10, "mailto": MAILTO}
    data, status = request_json(CONCEPTS_ENDPOINT, params)
    if not data or "results" not in data:
        raise RuntimeError(f"Could not fetch concept for {display_name}, status={status}")
    target = display_name.strip().lower()
    for c in data["results"]:
        dn = (c.get("display_name") or "").strip().lower()
        if dn == target:
            return c.get("id")
    return data["results"][0].get("id")

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def extract_author_ids_from_work(work):
    ids = []
    for a in work.get("authorships", []) or []:
        aid = (a.get("author") or {}).get("id")
        if aid:
            ids.append(aid)
    return ids

def fetch_works_for_author_batch(author_ids, concept_css_ids, concept_quant_ids):
    author_filter = "|".join(author_ids)
    css_filter = "|".join(concept_css_ids)
    quant_filter = "|".join(concept_quant_ids)

    base_filter = ",".join([
        f"author.id:{author_filter}",
        "cited_by_count:>10",
        "authors_count:<10",
        f"concept.id:{css_filter}",
        f"concept.id:{quant_filter}",
    ])

    cursor = "*"
    all_works = []
    while True:
        params = {
            "filter": base_filter,
            "per-page": 200,
            "cursor": cursor,
            "mailto": MAILTO,
        }
        data, status = request_json(WORKS_ENDPOINT, params)
        if not data:
            break
        results = data.get("results") or []
        all_works.extend(results)
        meta = data.get("meta") or {}
        next_cursor = meta.get("next_cursor")
        if not next_cursor:
            break
        cursor = next_cursor
        time.sleep(0.1)
    return all_works

df_auth = pd.read_csv(D1_AUTHORS_CSV)

df_auth = df_auth[df_auth["openalex_id"].notna()].copy()
df_auth["works_count"] = pd.to_numeric(df_auth["works_count"], errors="coerce")
df_auth = df_auth[df_auth["works_count"].between(5, 5000, inclusive="both")].copy()

author_ids = df_auth["openalex_id"].astype(str).tolist()
author_ids = [a.strip() for a in author_ids if a.strip()]

print("Authors in scope:", len(author_ids))

css_names = ["Sociology", "Psychology", "Economics", "Political science"]
quant_names = ["Mathematics", "Physics", "Computer science"]

concept_css_ids = [get_concept_id(n) for n in css_names]
concept_quant_ids = [get_concept_id(n) for n in quant_names]

print("CSS concept ids:", concept_css_ids)
print("Quant concept ids:", concept_quant_ids)

papers_rows = []
abstracts_rows = []
seen_work_ids = set()

total_batches = math.ceil(len(author_ids) / 25)
for b, batch_ids in enumerate(chunk(author_ids, 25), start=1):
    print(f"Batch {b}/{total_batches}, authors={len(batch_ids)}")
    works = fetch_works_for_author_batch(batch_ids, concept_css_ids, concept_quant_ids)
    print("  works fetched:", len(works))

    for w in works:
        wid = w.get("id")
        if not wid or wid in seen_work_ids:
            continue
        seen_work_ids.add(wid)

        publication_year = w.get("publication_year")
        cited_by_count = w.get("cited_by_count")
        w_title = w.get("title")
        abstract_inv = w.get("abstract_inverted_index")

        w_author_ids = extract_author_ids_from_work(w)

        papers_rows.append({
            "id": wid,
            "publication_year": publication_year,
            "cited_by_count": cited_by_count,
            "author_ids": json.dumps(w_author_ids, ensure_ascii=False),
        })

        abstracts_rows.append({
            "id": wid,
            "title": w_title,
            "abstract_inverted_index": json.dumps(abstract_inv, ensure_ascii=False) if abstract_inv is not None else None,
        })

    if b % 5 == 0:
        pd.DataFrame(papers_rows).to_csv(OUT_D2_PAPERS_CSV, index=False)
        pd.DataFrame(abstracts_rows).to_csv(OUT_D3_ABSTRACTS_CSV, index=False)
        print("  checkpoint saved")

    time.sleep(0.25)

df_d2 = pd.DataFrame(papers_rows).drop_duplicates(subset=["id"])
df_d3 = pd.DataFrame(abstracts_rows).drop_duplicates(subset=["id"])

df_d2.to_csv(OUT_D2_PAPERS_CSV, index=False)
df_d3.to_csv(OUT_D3_ABSTRACTS_CSV, index=False)

print("Done.")
print("D2 rows:", len(df_d2), "saved to", OUT_D2_PAPERS_CSV)
print("D3 rows:", len(df_d3), "saved to", OUT_D3_ABSTRACTS_CSV)
print("Unique works:", len(seen_work_ids))

Authors in scope: 1331
CSS concept ids: ['https://openalex.org/C144024400', 'https://openalex.org/C15744967', 'https://openalex.org/C162324750', 'https://openalex.org/C17744445']
Quant concept ids: ['https://openalex.org/C33923547', 'https://openalex.org/C121332964', 'https://openalex.org/C41008148']
Batch 1/54, authors=25
  works fetched: 160
Batch 2/54, authors=25
  works fetched: 599
Batch 3/54, authors=25
  works fetched: 270
Batch 4/54, authors=25
  works fetched: 335
Batch 5/54, authors=25
  works fetched: 183
  checkpoint saved
Batch 6/54, authors=25
  works fetched: 205
Batch 7/54, authors=25
  works fetched: 394
Batch 8/54, authors=25
  works fetched: 148
Batch 9/54, authors=25
  works fetched: 318
Batch 10/54, authors=25
  works fetched: 259
  checkpoint saved
Batch 11/54, authors=25
  works fetched: 426
Batch 12/54, authors=25
  works fetched: 714
Batch 13/54, authors=25
  works fetched: 334
Batch 14/54, authors=25
  works fetched: 181
Batch 15/54, authors=25
  works fetched

The abstracts csv was too big for github to upload at once so i split it into two files:

In [1]:
import pandas as pd

# Load full abstracts file
df = pd.read_csv("ic2s2_D3_abstracts.csv")

# Determine split point
midpoint = len(df) // 2

# Split dataframe
df_part1 = df.iloc[:midpoint]
df_part2 = df.iloc[midpoint:]

# Save split files
df_part1.to_csv("ic2s2_D3_abstracts_part1.csv", index=False)
df_part2.to_csv("ic2s2_D3_abstracts_part2.csv", index=False)

print("Part 1 rows:", len(df_part1))
print("Part 2 rows:", len(df_part2))

Part 1 rows: 7562
Part 2 rows: 7563
